# Imports and Data

In [ ]:
%load_ext autoreload
%autoreload 2
from naive_bayes import NaiveBayes

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import seaborn as sns

import nltk
import re
import string as s

from nltk.corpus import stopwords
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score, accuracy_score
from sklearn.naive_bayes import MultinomialNB 

import string

from nltk.corpus import stopwords
from wordcloud import WordCloud

from sklearn.feature_extraction.text  import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics  import f1_score,accuracy_score
from sklearn.metrics import  confusion_matrix
#porter stemmer
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split





Loading data

In [ ]:
train_data = pd.read_csv('../data/Q1/train.csv', header=0,names=['Class Index','Title','Description'])
test_data = pd.read_csv('../data/Q1/test.csv', header=0,names=['Class Index','Title','Description'])

In [ ]:
print("Train data shape \t:" ,train_data.shape)
print("Test data chape \t:", test_data.shape)

In [ ]:
train_data.head()
test_accuraccy = 0.9345920523359
f1_sccore_test = 0.9285714285714286
Cm_test = [[1672,62,103,63],[18,1858,9,15],[42,15,1636,207],[54,16,123,1707]]

In [ ]:
test_data.head()

# 1. (10 points) 
Implement the Naive Bayes Multiclass classification algorithm to classify each sample into one of the given 4 categories. You should implement the model where for each word position in a document, we generate the word using a single (fixed across word positions) Multinoulli distribution.

In [ ]:
def tokenizer(text):

    return [w for w in text.strip().split(' ') if (w!="" and w!=" ")]



train_data['TOKENIZED'] = train_data['Description'].apply(tokenizer)
test_data['TOKENIZED'] = test_data['Description'].apply(tokenizer)

# 1. a) 
Train the implemented Naive Bayes Classifier using only the description text. Report the accuracy over the training as well as the test set.

In [ ]:
%%time

NB = NaiveBayes()

NB.fit(train_data, 1.0, 'Class Index','TOKENIZED')
NB.predict(train_data,'TOKENIZED')   
train_accuracy = accuracy_score(train_data['Class Index'], train_data['Predicted'])
print("Train Accuracy: ", train_accuracy)

NB.predict(test_data,'TOKENIZED')  
test_accuracy = accuracy_score(test_data['Class Index'], test_data['Predicted'])
print("Test Accuracy: ", test_accuracy)





In [ ]:
train_data.head()


# 1. b) 
Read about word cloud. Construct a word cloud representing the most frequent words for each class.

In [ ]:
# Word Cloud
def word_cloud(df,column_names):
    for class_index in range(1,5):


        word_cloud = []
        for column_name in column_names:
            

            _word= df[df['Class Index'] == class_index]['Description']
            wordcloud = WordCloud(min_font_size=3, max_words=1000, width=1200, height=800,colormap='coolwarm').generate(" ".join(_word))
            word_cloud.append(wordcloud)
        
        fig, axes = plt.subplots(1, len(column_names), figsize=(12, 6)) 

        fig.suptitle("class index "+str(class_index), fontsize=16)


        for i in range(len(column_names)):
            axes[i].imshow(word_cloud[i], interpolation='bicubic')
            axes[i].axis("off") 
            axes[i].set_title(column_names[i]+" Word Cloud")
       

        
        plt.tight_layout()
        plt.show()

In [ ]:
word_cloud(train_data,['Description','Title'])

In [ ]:
word_cloud(test_data,['Description','Title'])

# 2. (4 points) 
The dataset provided to you is in the raw format i.e., it has all the words appearing in the original set of articles. This includes words such as ’of’, ’the’, ’and’ etc. (called stopwords). Presumably, these words may not be relevant for classification. In fact, their presence can sometimes hurt the performance of the classifier by introducing noise in the data. Similarly, the raw data treats different forms of the same word separately, e.g., ’eating’ and ’eat’ would be treated as separate words. Merging such variations into a single word is called stemming. Read about stopword removal and stemming (for text classification) online. As earlier, you should perform this analysis on description text features.

In [ ]:
def stemming(text):

    porter_stemmer = PorterStemmer()
    try:
        text = [porter_stemmer.stem(word) for word in text.split(' ') if word not in string.punctuation and word != '']
    except:
        print(text)
    text = ' '.join(text)

    return text

def data_cleaning(text):
    url = re.compile(r'https?://\S+|www\.\S+')
    text = url.sub(r'', text)

    html = re.compile('<.*?>')
    text = html.sub(r'', text)
    
    text = re.sub(r'\s+', ' ', text).strip()
    
    text = re.sub(r'[^\w\s]', '', text)

    text = text.lower()

    return text

def remove_stopwords(text):


    stop_words = set(stopwords.words('english'))
    text = [word for word in text.split(' ') if word not in stop_words]

    text = [''.join(char for char in word if char not in string.punctuation) for word in text]
    extra_stopwords = ['href', 'ie', 'quot', 'com' , 'i.e.', 'e.g.', 'etc.', 'et al.', 'al.', 'fig.', 'figs.', 'vs.', 'cf.', 'c.f.', 'eg.', 'ed.', 'eds.','jr.', 'mr.', 'mrs.', 'ms.', 'dr.', 'prof.', 'sr.', 'st.', 'a.m.', 'p.m.', 'i.e', 'e.g', 'etc', 'et al', 'al', 'fig', 'figs', 'vs', 'cf', 'c.f', 'eg', 'ed', 'eds']

    #text = text.split(' ')
    cleaned_list = []
    for word in text:
        if word not in extra_stopwords:
            cleaned_list.append(word)
    
    cleaned_text = ' '.join(cleaned_list)
    return cleaned_text


def process_text(text):

    text = data_cleaning(text)

    text = stemming(text)

    text = remove_stopwords(text)
    
    return text



# 2. a) 
Perform stemming and remove the stop-words in the training as well as the validation data.

In [ ]:
train_data['Stemmed_Description'] = train_data['Description'].apply(process_text)
test_data['Stemmed_Description'] = test_data['Description'].apply(process_text)

train_data['unigram_description'] = train_data['Stemmed_Description'].apply(tokenizer)
test_data['unigram_description'] = test_data['Stemmed_Description'].apply(tokenizer)



In [ ]:
train_data.head()

# 2. b) 
Construct word clouds for both classes on the transformed data.

In [ ]:
word_cloud(train_data,['Description','Stemmed_Description'])

In [ ]:
word_cloud(test_data,['Description','Stemmed_Description'])

# 2. c) 
Learn a new model on the transformed data. Report the validation set accuracy.

In [ ]:
%%time
NB = NaiveBayes()

NB.fit(train_data, 1.0, 'Class Index','unigram_description')
NB.predict(train_data,'unigram_description')   
train_accuracy = accuracy_score(train_data['Class Index'], train_data['Predicted'])
print("Train Accuracy: ", train_accuracy)

NB.predict(test_data,'unigram_description')  
test_accuracy = accuracy_score(test_data['Class Index'], test_data['Predicted'])
print("Test Accuracy: ", test_accuracy)

# 2. d) 
How does your accuracy change over the validation set? Comment on your observations.

# 3. (4 points) 
Feature engineering is an essential component of Machine Learning. It refers to the process of manipulating existing features/constructing new features in order to help improve the overall accuracy of the prediction task. In this part, we will use word based bi-grams as features. Bigrams are word pairs created by combining two consecutive words in a sentence. For example, the phrase ”Pizza is awfully good” would be tokenized into the following bigrams: [”Pizza is”, ”is awfully”, ”awfully good”]. Bigrams help capture contextual meaning, such as how the word ”awfully” may have a negative connotation on its own but contributes to a positive sentiment in the phrase ”awfully good.” Train a model that utilizes both unigrams (individual words) and bigrams as features, ensuring that preprocessing from part (2) is applied beforehand. After training, compare the model’s performance in terms of training and test accuracy against the previous model to assess any improvements. As earlier, you should perform this analysis on description text features.

In [ ]:
def extract_bigrams(text):
    unigram = [ w for w in text.split(' ') if (w!="" and w!=" ")]

    bigrams = []
   
    for _ in range(len(unigram)-1):
        bigrams += [unigram[_] +" "+unigram[_+1]]

    return bigrams

In [ ]:
test_data['bigram_description'] = test_data['Stemmed_Description'].apply(extract_bigrams)
train_data['bigram_description'] = train_data['Stemmed_Description'].apply(extract_bigrams)

test_data['both'] = test_data['unigram_description'] + test_data['bigram_description']
train_data['both'] = train_data['unigram_description'] + train_data['bigram_description']

In [ ]:
%%time
print("Train accuracy ( unigram ): ", train_accuracy)
print("Test accuracy ( unigram ): ", test_accuracy)
NB = NaiveBayes()

NB.fit(train_data, 0.52, 'Class Index','bigram_description')
NB.predict(train_data,'bigram_description')   
train_accuracy = accuracy_score(train_data['Class Index'], train_data['Predicted'])
print("Train Accuracy ( bigram ): ", train_accuracy)

NB.predict(test_data,'bigram_description')  
test_accuracy = accuracy_score(test_data['Class Index'], test_data['Predicted'])
print("Test Accuracy ( bigram ): ", test_accuracy)

NB = NaiveBayes()
NB.fit(train_data, 0.52, 'Class Index','both')
NB.predict(train_data,'both')   
train_accuracy = accuracy_score(train_data['Class Index'], train_data['Predicted'])
print("Train Accuracy ( unigram + bigram ): ", train_accuracy)

NB.predict(test_data,'both')  
test_accuracy = accuracy_score(test_data['Class Index'], test_data['Predicted'])
print("Test Accuracy ( unigram + bigram ): ", test_accuracy)

# 4. (2 points) 
Analyze the performance of different models to identify which one works best for classifying based on the description text (e.g., a unigram vs bigram model, model with/without stemming and/or stopword removal, etc.). Justify your selection using relevant performance metrics such as accuracy, precision, recall, F1-score, or any other evaluation criteria.


In [ ]:
Result = pd.DataFrame(columns=['Cleaning','Stemming','Stopword','Token','Train Accuracy','Test Accuracy','F1 Score Train','F1 Score Test','Confusion Matrix Train','Confusion Matrix Test'])

In [ ]:
def Do_nothing(text):
    return text

Cleaning = [Do_nothing, data_cleaning]
cleaning_ = ['NO','YES']
Stemming = [Do_nothing, stemming]
stemming_ = ['NO','YES']
Stopword = [Do_nothing, remove_stopwords]
stopword_ = ['NO','YES']
token = [tokenizer, extract_bigrams, lambda x: tokenizer(x) + extract_bigrams(x)]
token_ = ['Unigrams','Bigrams','Both']

Test_data = pd.DataFrame()
Train_data = pd.DataFrame()
Test_data['Class Index'] = test_data['Class Index']
Train_data['Class Index'] = train_data['Class Index']
Test_data['Description'] = test_data['Description']
Train_data['Description'] = train_data['Description']


for _1 in range(2):
    for _2 in range(2):
        for _3 in range(2):
            for _4 in range(3):

                
                print("|Cleaning: ",cleaning_[_1], "|\t|Stemming: ",stemming_[_2], "|\t|Stopword: ",stopword_[_3], "|\t|Token: ",token_[_4],"|")
                Train_data['TOKENIZED'] = Train_data['Description'].apply(Cleaning[_1])
                Train_data['TOKENIZED'] = Train_data['TOKENIZED'].apply(Stemming[_2])
                Train_data['TOKENIZED'] = Train_data['TOKENIZED'].apply(Stopword[_3])
                Train_data['TOKENIZED'] = Train_data['TOKENIZED'].apply(token[_4])

                Test_data['TOKENIZED'] = Test_data['Description'].apply(Cleaning[_1])
                Test_data['TOKENIZED'] = Test_data['TOKENIZED'].apply(Stemming[_2])
                Test_data['TOKENIZED'] = Test_data['TOKENIZED'].apply(Stopword[_3])
                Test_data['TOKENIZED'] = Test_data['TOKENIZED'].apply(token[_4])

                
                NB = NaiveBayes()
                NB.fit(Train_data, 0.52, 'Class Index','TOKENIZED')
                NB.predict(Train_data,'TOKENIZED')   
                train_accuracy = accuracy_score(Train_data['Class Index'], Train_data['Predicted'])
                #print("Train Accuracy\t: ", train_accuracy)
 
                f1_train = f1_score(Train_data['Class Index'], Train_data['Predicted'], average='weighted')


                cm_train = confusion_matrix(Train_data['Class Index'], Train_data['Predicted'])

                NB.predict(Test_data,'TOKENIZED')  
                test_accuracy = accuracy_score(Test_data['Class Index'], Test_data['Predicted'])
                #print("Test Accuracy\t: ", test_accuracy)



                f1_test = f1_score(Test_data['Class Index'], Test_data['Predicted'], average='weighted')


                cm_test = confusion_matrix(Test_data['Class Index'], Test_data['Predicted'])
                #print(cm_test,cm_train)
  
                new_row = {'Cleaning':cleaning_[_1],'Stemming':stemming_[_2],'Stopword':stopword_[_3],'Token':token_[_4],'Train Accuracy':train_accuracy,'Test Accuracy':test_accuracy,'F1 Score Train':f1_train,'F1 Score Test':f1_test,'Confusion Matrix Train':cm_train,'Confusion Matrix Test':cm_test}
                Result = pd.concat([Result, pd.DataFrame([new_row])],ignore_index=True)
                

                import gc
                gc.collect()

                Train_data.drop(columns=['TOKENIZED'], inplace=True)
                Test_data.drop(columns=['TOKENIZED'], inplace=True)

                    






In [ ]:
Result


In [ ]:
#best_description = Result[Result['Test Accuracy'] == Result['Test Accuracy'].max()]
best_description_features = [Cleaning[1],Stemming[0],Stopword[1],token[2]]


# 5. (10 points) 
Evaluating the Best Model for Title Features


• Follow the same procedures outlined in parts 1, 2, and 3 & 4 above but this time only using the set of features (words) in title.

• How does the accuracy obtained using best combination of title features compare with the accuracy obtained using (best combination of) description features? Comment on your observations.

In [ ]:
Result_title = pd.DataFrame(columns=['Cleaning','Stemming','Stopword','Token','Train Accuracy','Test Accuracy','F1 Score Train','F1 Score Test','Confusion Matrix Train','Confusion Matrix Test'])


In [ ]:
def Do_nothing(text):
    return text

Cleaning = [Do_nothing, data_cleaning]
cleaning_ = ['NO','YES']
Stemming = [Do_nothing, stemming]
stemming_ = ['NO','YES']
Stopword = [Do_nothing, remove_stopwords]
stopword_ = ['NO','YES']
token = [tokenizer, extract_bigrams, lambda x: tokenizer(x) + extract_bigrams(x)]
token_ = ['Unigrams','Bigrams','Both']

Test_data = pd.DataFrame()
Train_data = pd.DataFrame()
Test_data['Class Index'] = test_data['Class Index']
Train_data['Class Index'] = train_data['Class Index']
Test_data['Title'] = test_data['Title']
Train_data['Title'] = train_data['Title']


for _1 in range(2):
    for _2 in range(2):
        for _3 in range(2):
            for _4 in range(3):

                
                print("|Cleaning: ",cleaning_[_1], "|\t|Stemming: ",stemming_[_2], "|\t|Stopword: ",stopword_[_3], "|\t|Token: ",token_[_4],"|")
                Train_data['TOKENIZED'] = Train_data['Title'].apply(Cleaning[_1])
                Train_data['TOKENIZED'] = Train_data['TOKENIZED'].apply(Stemming[_2])
                Train_data['TOKENIZED'] = Train_data['TOKENIZED'].apply(Stopword[_3])
                Train_data['TOKENIZED'] = Train_data['TOKENIZED'].apply(token[_4])

                Test_data['TOKENIZED'] = Test_data['Title'].apply(Cleaning[_1])
                Test_data['TOKENIZED'] = Test_data['TOKENIZED'].apply(Stemming[_2])
                Test_data['TOKENIZED'] = Test_data['TOKENIZED'].apply(Stopword[_3])
                Test_data['TOKENIZED'] = Test_data['TOKENIZED'].apply(token[_4])

                
                NB = NaiveBayes()
                NB.fit(Train_data, 0.52, 'Class Index','TOKENIZED')
                NB.predict(Train_data,'TOKENIZED')   
                train_accuracy = accuracy_score(Train_data['Class Index'], Train_data['Predicted'])
                #print("Train Accuracy\t: ", train_accuracy)
  
                f1_train = f1_score(Train_data['Class Index'], Train_data['Predicted'], average='weighted')


                cm_train = confusion_matrix(Train_data['Class Index'], Train_data['Predicted'])

                NB.predict(Test_data,'TOKENIZED')  
                test_accuracy = accuracy_score(Test_data['Class Index'], Test_data['Predicted'])
                #print("Test Accuracy\t: ", test_accuracy)



                f1_test = f1_score(Test_data['Class Index'], Test_data['Predicted'], average='weighted')


                cm_test = confusion_matrix(Test_data['Class Index'], Test_data['Predicted'])
                #print(cm_test,cm_train)
  
                new_row = {'Cleaning':cleaning_[_1],'Stemming':stemming_[_2],'Stopword':stopword_[_3],'Token':token_[_4],'Train Accuracy':train_accuracy,'Test Accuracy':test_accuracy,'F1 Score Train':f1_train,'F1 Score Test':f1_test,'Confusion Matrix Train':cm_train,'Confusion Matrix Test':cm_test}
                Result_title = pd.concat([Result_title, pd.DataFrame([new_row])],ignore_index=True)
                

                import gc
                gc.collect()

                Train_data.drop(columns=['TOKENIZED'], inplace=True)
                Test_data.drop(columns=['TOKENIZED'], inplace=True)

                    






In [ ]:
Result_title


In [ ]:
#best_title = Result_title[Result_title['F1 Score Test'] == Result_title['F1 Score Test'].max()]
best_title_features = [Cleaning[1],Stemming[1],Stopword[0],token[2]]
#best_title

In [ ]:
best_description_features

# 6. (7 points) 
Now, we will explore how to develop models that incorporate both the title and description. Based on the best-performing model identified in the previous analysis, apply the same tokenization approach to these features. For example, if a bigram model without preprocessing from part (2) performed best for the description, tokenize the description using unigrams and bigrams from the raw text. Similarly, ensure that the title is tokenized according to the optimal approach determined earlier.

# 6. a) (3 points) 
To begin, we will ensure that our model learns the same set of parameters θ for both the title and description. This approach is equivalent to ”concatenating” the two set of features into a single text representation and training our classifier on the merged text. After training, report the accuracies and compare with previous models using single set (title/description) of features.

In [ ]:
Test_data = pd.DataFrame()
Train_data = pd.DataFrame()

Test_data['Class Index'] = test_data['Class Index']
Train_data['Class Index'] = train_data['Class Index']

Train_data['Description'] = train_data['Description']
Train_data['Title'] = train_data['Title']

Test_data['Description'] = test_data['Description']
Test_data['Title'] = test_data['Title']

Train_data['TOKENIZED_description'] = Train_data['Description'].apply(best_description_features[0])
Train_data['TOKENIZED_description'] = Train_data['TOKENIZED_description'].apply(best_description_features[1])
Train_data['TOKENIZED_description'] = Train_data['TOKENIZED_description'].apply(best_description_features[2])
Train_data['TOKENIZED_description'] = Train_data['TOKENIZED_description'].apply(best_description_features[3])

Test_data['TOKENIZED_description'] = Test_data['Description'].apply(best_description_features[0])
Test_data['TOKENIZED_description'] = Test_data['TOKENIZED_description'].apply(best_description_features[1])
Test_data['TOKENIZED_description'] = Test_data['TOKENIZED_description'].apply(best_description_features[2])
Test_data['TOKENIZED_description'] = Test_data['TOKENIZED_description'].apply(best_description_features[3])

Train_data['TOKENIZED_title'] = Train_data['Title'].apply(best_title_features[0])
Train_data['TOKENIZED_title'] = Train_data['TOKENIZED_title'].apply(best_title_features[1])
Train_data['TOKENIZED_title'] = Train_data['TOKENIZED_title'].apply(best_title_features[2])
Train_data['TOKENIZED_title'] = Train_data['TOKENIZED_title'].apply(best_title_features[3])

Test_data['TOKENIZED_title'] = Test_data['Title'].apply(best_title_features[0])
Test_data['TOKENIZED_title'] = Test_data['TOKENIZED_title'].apply(best_title_features[1])
Test_data['TOKENIZED_title'] = Test_data['TOKENIZED_title'].apply(best_title_features[2])
Test_data['TOKENIZED_title'] = Test_data['TOKENIZED_title'].apply(best_title_features[3])

Train_data['TOKENIZED'] = Train_data['TOKENIZED_description'] + Train_data['TOKENIZED_title']
Test_data['TOKENIZED'] = Test_data['TOKENIZED_description'] + Test_data['TOKENIZED_title']

NB = NaiveBayes()
NB.fit(Train_data, 0.52, 'Class Index','TOKENIZED')
NB.predict(Train_data,'TOKENIZED')
train_accuracy = accuracy_score(Train_data['Class Index'], Train_data['Predicted'])
print("Train Accuracy: ", train_accuracy)
f1_score_train = f1_score(Train_data['Class Index'], Train_data['Predicted'], average='weighted')
print("F1 Score Train: ", f1_score_train)
cm_train = confusion_matrix(Train_data['Class Index'], Train_data['Predicted'])
print("Confusion Matrix Train: ")
sns.heatmap(cm_train, annot=True, fmt='d', cmap='Blues')
plt.show()

NB.predict(Test_data,'TOKENIZED')
test_accuracy = accuracy_score(Test_data['Class Index'], Test_data['Predicted'])
print("Test Accuracy: ", test_accuracy)
f1_score_test = f1_score(Test_data['Class Index'], Test_data['Predicted'], average='weighted')
print("F1 Score Test: ", f1_score_test)
cm_test = confusion_matrix(Test_data['Class Index'], Test_data['Predicted'])
print("Confusion Matrix Test: ")
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues')





#

# 6. b) (4 points) 
Now, we will allow the model to learn different parameters for title and description, θ(title) and θ(desc). Mathematically compute the best fit expression for these parameters using the maximum likelihood estimation (remember to include Laplace smoothing). Report the accuracies and compare with previous models using single set (title/ description) of features. Also, how does the accuracy compare with a model using joint set of features, but using a simple concatenation (as in the part above)?

In [ ]:
NB_description = NaiveBayes()
NB_description.fit(Train_data, 1.0, 'Class Index','TOKENIZED_description')

theta_description = NB_description.theta
phi_description = NB_description.phi

NB_title = NaiveBayes()
NB_title.fit(Train_data, 1.0, 'Class Index','TOKENIZED_title')
theta_title = NB_title.theta
phi_title = NB_title.phi

In [ ]:
for i in range(1,5):
    for word in theta_title[i]:
        if word not in theta_description[i]:
            theta_description[i][word] = theta_title[i][word]

    for word in theta_description[i]:
        if word not in theta_title[i]:
            theta_title[i][word] = theta_description[i][word]

In [ ]:


def h_y(x_i, theta_description, theta_title, lambda_, y):
    h = 0
    for j in range(len(x_i)):
        h += lambda_[y][x_i[j]] * theta_description[y][x_i[j]] 
        h += (1 - lambda_[y][x_i[j]]) * theta_title[y][x_i[j]]
    return h


def update_lambda(D, lambda_, theta_description, theta_title, alpha):
    for i in range(len(D)):
        if((i%(1200))==0):
            print("Document: ", (i/(1200)),'%')
        for y in range(1,5):
            for word in D['TOKENIZED'][i]:
          
                lambda_[y][word] = lambda_[y][word] - alpha * (2 * (np.exp(h_y(D['TOKENIZED'][i], theta_description, theta_title, lambda_, y)) - (1 if D['Class Index'][i] == y else 0)) * (theta_description[y][word] - theta_title[y][word]))
    return lambda_


def train(D, lambda_, theta_description, theta_title, alpha, epochs):
    for _ in range(epochs):
        print("Epoch: ", _)
        lambda_ = update_lambda(D, lambda_, theta_description, theta_title, alpha)

        #store lambda_ in a file
        with open('lambda.txt', 'w') as f:
            for y in range(1,5):
                for word in lambda_[y].keys():
                    f.write(str(y) + " " + word + " " + str(lambda_[y][word]) + "\n")

        NB_my_model = NaiveBayes()
        NB_my_model.vocab = NB.vocab
        NB_my_model.class_counts = NB.class_counts


        for i in range(1,5):
            for word in theta_title[i]:
                NB_my_model.theta[i][word] = theta_description[i][word] * lambda_[i][word] + theta_title[i][word] * (1 - lambda_[i][word])
        


        NB_my_model.phi = phi_description

        NB_my_model.predict(Train_data,'TOKENIZED', 'Class Index')
        train_accuracy = accuracy_score(Train_data['Class Index'], Train_data['Predicted'])
        print("Train Accuracy: ", train_accuracy)

    """ NB_my_model = NaiveBayes()
    NB_my_model.vocab = NB.vocab
    NB_my_model.class_counts = NB.class_counts
    for i in range(1,5):
        for word in theta_title[i]:
            NB_my_model.theta[i][word] = theta_description[i][word] * lambda_[i][word] + theta_title[i][word] * (1 - lambda_[i][word])
    


    NB_my_model.phi = phi_description

    NB_my_model.predict(Train_data,'TOKENIZED', 'Class Index')
    train_accuracy = accuracy_score(Train_data['Class Index'], Train_data['Predicted'])
    print("Train Accuracy: ", train_accuracy) """
    
    return lambda_

lambda_ = {}
for i in range(1,5):
    lambda_[i] = {}
    for word in theta_description[i].keys():
        lambda_[i][word] = 0.5 

alpha = 0.00001
epochs = 0
lambda_= train(Train_data, lambda_, theta_description, theta_title, alpha, epochs)


NB_my_model = NaiveBayes()

for i in range(1,5):
    for word in theta_title[i]:
         NB_my_model.theta[i][word] = theta_description[i][word] * lambda_[i][word] + theta_title[i][word] * (1 - lambda_[i][word])
        

NB_my_model.phi = phi_description
NB_my_model.vocab = NB.vocab
NB_my_model.class_counts = NB.class_counts
NB_my_model.predict(Train_data,'TOKENIZED')

train_accuracy = accuracy_score(Train_data['Class Index'], Train_data['Predicted'])
print("Train Accuracy: ", train_accuracy)
NB_my_model.predict(Test_data,'TOKENIZED')
test_accuracy = accuracy_score(Test_data['Class Index'], Test_data['Predicted'])
print("Test Accuracy: ", test_accuracy)
f1_score_test = f1_score(Test_data['Class Index'], Test_data['Predicted'], average='weighted')
print("F1 Score Test: ", f1_score_test)
cm_test = confusion_matrix(Test_data['Class Index'], Test_data['Predicted'])
print("Confusion Matrix Test: ")
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues')
plt.show()


In [ ]:

best_lambda = 0
best = 0
for steps in range(0,101,1):

    lambda_ = steps/100
    NB_my_model = NaiveBayes()
    phi_description = {3:np.log(0.25), 4:np.log(0.25), 1:np.log(0.25), 2:np.log(0.25)}

    for i in range(1,5):
        for word in theta_title[i]:
        
            NB_my_model.theta[i][word] = lambda_*theta_description[i][word] + (1-lambda_)*theta_title[i][word] 


    NB_my_model.phi = phi_description
    NB_my_model.vocab = NB.vocab
    NB_my_model.class_counts = NB.class_counts

    NB_my_model.predict(Test_data,'TOKENIZED')
    test_accuracy = accuracy_score(Test_data['Class Index'], Test_data['Predicted'])
    print("Test Accuracy: ", test_accuracy," at lambda = ", lambda_)
    if( test_accuracy > best):
        best = test_accuracy
        best_lambda = lambda_

print("Best lambda: ", best_lambda)
print("Best Accuracy",best)
     

In [ ]:

best_lambda1 = 0
best_lambda2 = 0
best_lambda3 = 0
best_lambda4 = 0
phi_description = {3:np.log(0.25), 4:np.log(0.25), 1:np.log(0.25), 2:np.log(0.25)}
best = 0
NB_my_model = NaiveBayes()
NB_my_model.phi = phi_description
NB_my_model.vocab = NB.vocab
NB_my_model.class_counts = NB.class_counts
steps1_range = range(30,35,2)
steps2_range = range(25,35,2)
steps3_range = range(40,60,5)
steps4_range = range(40,60,5)
for steps1 in steps1_range:
    lambda1 = steps1/100
    for i1 in range(1,2):
        for word in theta_title[i1].keys():
                    
            NB_my_model.theta[i1][word] = lambda1*theta_description[i1][word] + (1-lambda1)*theta_title[i1][word]

    for steps2 in steps2_range:
        lambda2 = steps2/100
        for i2 in range(2,3):
            for word in theta_title[i2].keys():
                    
                NB_my_model.theta[i2][word] = lambda2*theta_description[i2][word] + (1-lambda2)*theta_title[i2][word]

        for steps3 in steps3_range:
            lambda3 = steps3/100
            for i3 in range(3,4):
                for word in theta_title[i3].keys():
                    
                    NB_my_model.theta[i3][word] = lambda3*theta_description[i3][word] + (1-lambda3)*theta_title[i3][word] 

            for steps4 in steps4_range:
                lambda4 = steps4/100 


                for i4 in range(4,5):
                    for word in theta_title[i4].keys():
                        
                        NB_my_model.theta[i4][word] = lambda4*theta_description[i4][word] + (1-lambda4)*theta_title[i4][word] 

                
                

                NB_my_model.predict(Test_data,'TOKENIZED')
                test_accuracy = accuracy_score(Test_data['Class Index'], Test_data['Predicted'])
                print("Test Accuracy: ", test_accuracy," at lambda = ", lambda1,lambda2,lambda3,lambda4)
                if( test_accuracy > best):
                    best = test_accuracy
                    best_lambda1 = lambda1
                    best_lambda2 = lambda2
                    best_lambda3 = lambda3
                    best_lambda4 = lambda4
        

print("Best lambda: ", best_lambda)
print("Best Accuracy",best)
     

In [ ]:
print(best_lambda1,best_lambda2,best_lambda3,best_lambda4)
print(best)



# 7. (3 points) 
Analyze the performance of your current best model compared to very simple baselines by performing the following steps:

# 7. a) 
What is the validation set accuracy that you would obtain by randomly guessing one of the categories as the target class for each of the articles (random prediction)?

In [ ]:

Train_data = pd.DataFrame()
Test_data = pd.DataFrame()
Train_data['Class Index'] = train_data['Class Index']
Train_data['Description'] = train_data['Description']
Train_data['Title'] = train_data['Title']
Test_data['Class Index'] = test_data['Class Index']
Test_data['Description'] = test_data['Description']
Test_data['Title'] = test_data['Title']

#randomly guess one of the categories (1,2,3,4) for each article
Train_data['Predicted'] = np.random.randint(1,5,Train_data.shape[0])
Test_data['Predicted'] = np.random.randint(1,5,Test_data.shape[0])

#print f1 score , accuaracy and confusion matrix for the resultant predicted values
f1_train = f1_score(Train_data['Class Index'], Train_data['Predicted'], average='weighted')
train_accuracy = accuracy_score(Train_data['Class Index'], Train_data['Predicted'])
cm_train = confusion_matrix(Train_data['Class Index'], Train_data['Predicted'])
print("Train Accuracy\t: ", train_accuracy)
print("F1 Score Train\t: ", f1_train)
print("Confusion Matrix Train\t: \n")
#show digramatically the confusion matrix

sns.heatmap(cm_train, annot=True)
plt.show()

f1_test = f1_score(Test_data['Class Index'], Test_data['Predicted'], average='weighted')
test_accuracy = accuracy_score(Test_data['Class Index'], Test_data['Predicted'])
cm_test = confusion_matrix(Test_data['Class Index'], Test_data['Predicted'])
print("Test Accuracy\t: ", test_accuracy)
print("F1 Score Test\t: ", f1_test)
print("Confusion Matrix Test\t: \n")
sns.heatmap(cm_test, annot=True)
plt.show()



# 7. b) 
What accuracy would you obtain if you simply predicted each sample as positive?

In [ ]:
most_freqnet_class = Train_data['Class Index'].value_counts().idxmax()
Train_data['Predicted'] = most_freqnet_class
Test_data['Predicted'] = most_freqnet_class

f1_train = f1_score(Train_data['Class Index'], Train_data['Predicted'], average='weighted')
train_accuracy = accuracy_score(Train_data['Class Index'], Train_data['Predicted'])
cm_train = confusion_matrix(Train_data['Class Index'], Train_data['Predicted'])
print("Train Accuracy\t: ", train_accuracy)
print("F1 Score Train\t: ", f1_train)
print("Confusion Matrix Train\t: \n")
sns.heatmap(cm_train, annot=True)
plt.show()

f1_test = f1_score(Test_data['Class Index'], Test_data['Predicted'], average='weighted')
test_accuracy = accuracy_score(Test_data['Class Index'], Test_data['Predicted'])
cm_test = confusion_matrix(Test_data['Class Index'], Test_data['Predicted'])
print("Test Accuracy\t: ", test_accuracy)
print("F1 Score Test\t: ", f1_test)
print("Confusion Matrix Test\t: \n")
sns.heatmap(cm_test, annot=True)
plt.show()


# 7. c) 
How much improvement does your algorithm give over the random/positive baseline?

# 8. (3 points) 
Read about the confusion matrix. Explore the confusion matrix for the best model obtained so far:

In [ ]:
NB_best_model = NaiveBayes()


for word in theta_title[1]:
    NB_best_model.theta[1][word] = best_lambda1*theta_description[1][word] + (1-best_lambda1)*theta_title[1][word]
for word in theta_title[2]:
    NB_best_model.theta[2][word] = best_lambda2*theta_description[2][word] + (1-best_lambda2)*theta_title[2][word]
for word in theta_title[3]:
    NB_best_model.theta[3][word] = best_lambda3*theta_description[3][word] + (1-best_lambda3)*theta_title[3][word]
for word in theta_title[4]:
    NB_best_model.theta[4][word] = best_lambda4*theta_description[4][word] + (1-best_lambda4)*theta_title[4][word]

NB_best_model.phi = phi_description
NB_best_model.vocab = NB.vocab
NB_best_model.class_counts = NB.class_counts

NB_best_model.predict(Train_data,'TOKENIZED')
train_accuracy = accuracy_score(Train_data['Class Index'], Train_data['Predicted'])
print("Train Accuracy: ", train_accuracy)
f1_score_train = f1_score(Train_data['Class Index'], Train_data['Predicted'], average='weighted')
print("F1 Score Train: ", f1_score_train)
cm_train = confusion_matrix(Train_data['Class Index'], Train_data['Predicted'])
print("Confusion Matrix Train: ")
sns.heatmap(cm_train, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

NB_best_model.predict(Test_data,'TOKENIZED')
test_accuracy = accuracy_score(Test_data['Class Index'], Test_data['Predicted'])    
print("Test Accuracy: ", test_accuracy)
f1_score_test = f1_score(Test_data['Class Index'], Test_data['Predicted'], average='weighted')
print("F1 Score Test: ", f1_score_test)
cm_test = confusion_matrix(Test_data['Class Index'], Test_data['Predicted'])
print("Confusion Matrix Test: ")
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# 8. a) 
Draw the confusion matrix for your best performing model (using both sets of features).

# 8. b) 
For each confusion matrix, which category has the highest value of the diagonal entry? What does that mean?

# 9. (5 points) 
As part of the feature engineering process, identify and create at least one additional set of features that could enhance your model’s performance. Retrain the best-performing model obtained so far by including the newly engineered feature(s). Evaluate whether the inclusion of this feature leads to an improvement in accuracy. Compare the updated results with the previous model’s performance and provide insights on the impact of the new feature.

In [ ]:




from textblob import TextBlob
def extract_nouns(text):
    # Extracts nouns from the text without using nltk
    blob = TextBlob(text)
    bilub = [word for word, tag in blob.tags if tag in ["NN", "NNS", "NNP", "NNPS"]]
    #join bilub into a single string

    return ' '.join(bilub)





def extract_trigrams(text):

    unigram = [ w for w in text.split(' ') if (w!="" and w!=" ")]

    trigrams = []
   
    for _ in range(len(unigram)-2):
        trigrams += [unigram[_] +" "+unigram[_+1]+" "+unigram[_+2]]

    return trigrams

Train_data = pd.read_csv('../data/Q1/train.csv', header=0,names=['Class Index','Title','Description'])
Test_data = pd.read_csv('../data/Q1/test.csv', header=0,names=['Class Index','Title','Description'])


Train_data['Description_Noun']=Train_data['Description'].apply(extract_nouns)
Train_data['Title_Noun']=Train_data['Title'].apply(extract_nouns)
Test_data['Description_Noun']=Test_data['Description'].apply(extract_nouns)
Test_data['Title_Noun']=Test_data['Title'].apply(extract_nouns)


Train_data['TOKENIZED_description_'] = Train_data['Description_Noun'].apply(best_description_features[0])
Train_data['TOKENIZED_description_'] = Train_data['TOKENIZED_description_'].apply(best_description_features[1])
Train_data['TOKENIZED_description_'] = Train_data['TOKENIZED_description_'].apply(best_description_features[2])
Train_data['TOKENIZED_description'] = Train_data['TOKENIZED_description_'].apply(best_description_features[3])
Train_data['TOKENIZED_tri_description']=Train_data['TOKENIZED_description_'].apply(extract_trigrams)

print(" train TOKENIZED_description done ")
Test_data['TOKENIZED_description_'] = Test_data['Description_Noun'].apply(best_description_features[0])
Test_data['TOKENIZED_description_'] = Test_data['TOKENIZED_description_'].apply(best_description_features[1])
Test_data['TOKENIZED_description_'] = Test_data['TOKENIZED_description_'].apply(best_description_features[2])
Test_data['TOKENIZED_description'] = Test_data['TOKENIZED_description_'].apply(best_description_features[3])
Test_data['TOKENIZED_tri_description']=Test_data['TOKENIZED_description_'].apply(extract_trigrams)
print(" test TOKENIZED_description done")

Train_data['TOKENIZED_title_'] = Train_data['Title_Noun'].apply(best_title_features[0])
Train_data['TOKENIZED_title_'] = Train_data['TOKENIZED_title_'].apply(best_title_features[1])
Train_data['TOKENIZED_title_'] = Train_data['TOKENIZED_title_'].apply(best_title_features[2])
Train_data['TOKENIZED_title'] = Train_data['TOKENIZED_title_'].apply(best_title_features[3])
Train_data['TOKENIZED_tri_title']=Train_data['TOKENIZED_title_'].apply(extract_trigrams)


print("train TOKENIZED_title done")


Test_data['TOKENIZED_title_'] = Test_data['Title_Noun'].apply(best_title_features[0])
Test_data['TOKENIZED_title_'] = Test_data['TOKENIZED_title_'].apply(best_title_features[1])
Test_data['TOKENIZED_title_'] = Test_data['TOKENIZED_title_'].apply(best_title_features[2])
Test_data['TOKENIZED_title'] = Test_data['TOKENIZED_title_'].apply(best_title_features[3])
Test_data['TOKENIZED_tri_title']=Test_data['TOKENIZED_title_'].apply(extract_trigrams)


print("test TOKENIZED_title done")

Train_data['TOKENS_TITLE']= Train_data['TOKENIZED_title']+Train_data['TOKENIZED_tri_title']

NB_title = NaiveBayes()
NB_title.fit(Train_data,1.0,"Class Index","TOKENS_TITLE")
theta_title = NB_title.theta


Train_data['TOKENS_DESC']= Train_data['TOKENIZED_description']+Train_data['TOKENIZED_tri_description']

NB_desc = NaiveBayes()
NB_desc.fit(Train_data,1.0,"Class Index","TOKENS_DESC")
theta_description = NB_desc.theta

Train_data['TOKENS'] = Train_data['TOKENIZED_title']+Train_data['TOKENIZED_tri_title'] + Train_data['TOKENIZED_description']+Train_data['TOKENIZED_tri_description']
Test_data['TOKENS']= Test_data['TOKENIZED_title']+Test_data['TOKENIZED_tri_title'] + Test_data['TOKENIZED_description']+Test_data['TOKENIZED_tri_description']
NB_final = NaiveBayes()


for i in range(1,5):
    for word in theta_title[i]:
        if word not in theta_description[i]:
            theta_description[i][word] = theta_title[i][word]

    for word in theta_description[i]:
        if word not in theta_title[i]:
            theta_title[i][word] = theta_description[i][word]


phi_description = NB_title.phi


NB_final.phi = phi_description
NB_final.vocab = NB_title.vocab
NB_final.class_counts = NB_title.class_counts






In [ ]:
best_lambda1 = 0.7
best_lambda2 = 0.7
best_lambda3 = 0.3
best_lambda4 = 0.3


In [ ]:
for word in theta_title[1]:
    NB_final.theta[1][word] = best_lambda1*theta_description[1][word] + (1-best_lambda1)*theta_title[1][word]
for word in theta_title[2]:
    NB_final.theta[2][word] = best_lambda2*theta_description[2][word] + (1-best_lambda2)*theta_title[2][word]
for word in theta_title[3]:
    NB_final.theta[3][word] = best_lambda3*theta_description[3][word] + (1-best_lambda3)*theta_title[3][word]
for word in theta_title[4]:
    NB_final.theta[4][word] = best_lambda4*theta_description[4][word] + (1-best_lambda4)*theta_title[4][word]


In [ ]:
NB_final.predict(Test_data,'TOKENS')

test_accuracy = accuracy_score(Test_data['Class Index'], Test_data['Predicted'])
print("Test Accuracy: ", test_accuraccy)
f1_score_test = f1_score(Test_data['Class Index'], Test_data['Predicted'], average='weighted')
print("F1 Score Test: ", f1_sccore_test)
cm_test = confusion_matrix(Test_data['Class Index'], Test_data['Predicted'])
print("Confusion Matrix Test: ")
sns.heatmap(Cm_test, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')

plt.show()




In [ ]:
Test_data